# Geometry-branch dropout

## Research question

This jointly removes all gated UVD channels for 30% of training examples, without inverted scaling. It tests whether the model becomes less dependent on perfect object-mask geometry.

The visual encoder (DINOv2 ViT-S/14) and text encoder (OpenCLIP ViT-B/32) are loaded strictly from local checkpoints and remain frozen. The trainable projection, geometry-control, and decoder components are optimized from scratch for this experiment.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "final_training_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "final_training").is_dir():
    raise FileNotFoundError("Run this notebook from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
print("Project root:", PROJECT_ROOT)
print("Training run:", RUN_ID)


## Training and stopping rule

Training uses mixed precision on CUDA, a physical batch size of 8 with two-step gradient accumulation (effective batch 16), AdamW, gradient clipping, and a validation-controlled learning-rate schedule. The maximum is 30 epochs. Training cannot stop before epoch 8 and stops after five consecutive epochs without a validation-IoU improvement greater than 0.001.

`last.pt` is saved after every epoch for interruption recovery. `best.pt` and `ui_model.pt` are selected only by `validation_seen` IoU. Test metrics never control training or checkpoint selection.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Submit scripts/submit_full_training_fau.slurm on Alex.")
print("GPU:", torch.cuda.get_device_name(0))

from final_training.training_core import run_experiment

summary = run_experiment("geometry_dropout")
summary

## Produced evidence

This notebook writes its checkpoint to `trained_points/<run-id>/geometry_dropout/` and its metrics, per-example predictions, configuration, training curves, and unseen qualitative examples to `final_training_results/<run-id>/geometry_dropout/`. Re-execution with the same run ID resumes from the last completed epoch.